# Setup Data

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools

ticker = "^GSPC" # L'ETF du S&P 500
start_date = "1985-01-01"
end_date = "2026-01-01"

print(f"📥 Téléchargement des données pour {ticker}...")
# On télécharge les données journalières depuis Yahoo Finance
df = yf.download(ticker, start=start_date, end=end_date, interval="1mo")

# On s'assure que l'index est bien un format Date et on garde le prix de clôture
df = df[['Close']].copy()
# yfinance renvoie parfois un MultiIndex sur les colonnes, on l'aplatit au cas où
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Paramètres du Levier
leverage = 1.0
annual_margin_rate = 0.05

# Rendements de base
df['Market_Return'] = df['Close'].pct_change()
daily_borrowing_cost = (annual_margin_rate / 12) * (leverage - 1.0)
df['Leveraged_Return'] = (df['Market_Return'] * leverage) - daily_borrowing_cost

print(f"nombre de données: {len(df)}")

📥 Téléchargement des données pour ^GSPC...


[*********************100%***********************]  1 of 1 completed

nombre de données: 492


# Grid Search : Trend Following

In [2]:
# On teste différentes fenêtres rapides et lentes
ma_shorts = range(1, 50, 1)
ma_longs = range(1, 50, 1)

print(f"🚀 Lancement de la Grid Search ({len(ma_shorts) * len(ma_longs)} combinaisons)...")
results = []

🚀 Lancement de la Grid Search (2401 combinaisons)...


# Backtesting a Trend Following Strategy

In [3]:
print(ma_shorts)
print(ma_longs)
for short_w, long_w in itertools.product(ma_shorts, ma_longs):
    # On ignore les combinaisons absurdes (la rapide doit être plus petite que la lente)
    if short_w >= long_w:
        continue
        
    temp_df = df.copy()
    
    # Calcul des Moyennes Mobiles
    temp_df['MA_short'] = temp_df['Close'].rolling(window=short_w).mean()
    temp_df['MA_long'] = temp_df['Close'].rolling(window=long_w).mean()
    
    # Ta condition stricte : Prix > MA_Rapide ET MA_Rapide > MA_Lente
    condition1 = (temp_df['Close'] > temp_df['MA_short']) & (temp_df['MA_short'] > temp_df['MA_long'])
    condition2 = (temp_df['Close'] > temp_df['MA_short']) & (temp_df['MA_short'] < temp_df['MA_long'])

    temp_df['Signal'] = np.where(condition1 | condition2, 1, 0)
    
    # Calcul du rendement de la stratégie avec le fameux shift(1)
    temp_df['Strategy_Return'] = temp_df['Signal'].shift(1) * temp_df['Leveraged_Return']
    
    # On retire les jours au début où les moyennes mobiles n'étaient pas encore calculables
    temp_df = temp_df.dropna()
    
    # --- Calcul des Métriques ---
    n_years = (temp_df.index[-1] - temp_df.index[0]).days / 365.25
    
    # CAGR
    strat_cum = (1 + temp_df['Strategy_Return']).cumprod().iloc[-1]
    cagr = (strat_cum ** (1 / n_years)) - 1
    
    # Sharpe Ratio
    mean_ret = temp_df['Strategy_Return'].mean()
    std_ret = temp_df['Strategy_Return'].std()
    sharpe = (mean_ret / std_ret) * np.sqrt(12) if std_ret > 0 else 0
    
    # Max Drawdown
    cum_returns = (1 + temp_df['Strategy_Return']).cumprod()
    rolling_max = cum_returns.cummax()
    max_dd = ((cum_returns - rolling_max) / rolling_max).min()
    
    # Sauvegarde du résultat
    results.append({
        'MA_Rapide': short_w,
        'MA_Lente': long_w,
        'CAGR': cagr * 100,          # En pourcentage
        'Sharpe_Ratio': sharpe,
        'Max_Drawdown': max_dd * 100 # En pourcentage
    })

range(1, 50)
range(1, 50)


# Result of the trend following strategy

In [4]:
max_window = max(ma_longs)
bench_df = df.iloc[max_window:].copy()
n_years_bench = (bench_df.index[-1] - bench_df.index[0]).days / 365.25

bench_cum = (1 + bench_df['Market_Return']).cumprod().iloc[-1]
bench_cagr = (bench_cum ** (1 / n_years_bench)) - 1
bench_sharpe = (bench_df['Market_Return'].mean() / bench_df['Market_Return'].std()) * np.sqrt(12)

bench_cum_returns = (1 + bench_df['Market_Return']).cumprod()
bench_max_dd = ((bench_cum_returns - bench_cum_returns.cummax()) / bench_cum_returns.cummax()).min()

# Création du DataFrame final et tri par le meilleur CAGR
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='CAGR', ascending=False).reset_index(drop=True)

print("\n" + "="*60)
print(f"🎯 BENCHMARK (S&P 500 Buy & Hold) : CAGR {bench_cagr*100:.2f}% | Sharpe {bench_sharpe:.2f} | Max DD {bench_max_dd*100:.2f}%")
print("="*60)
print("\n🏆 TOP 10 DES MEILLEURES COMBINAISONS (Avec Levier 2x) :")
print(results_df.head(10).round(2).to_string(index=False))
print("="*60)


🎯 BENCHMARK (S&P 500 Buy & Hold) : CAGR 8.89% | Sharpe 0.66 | Max DD -52.56%

🏆 TOP 10 DES MEILLEURES COMBINAISONS (Avec Levier 2x) :
 MA_Rapide  MA_Lente  CAGR  Sharpe_Ratio  Max_Drawdown
        43        44  9.11          0.77         -25.4
        43        47  9.04          0.77         -25.4
        43        48  9.02          0.76         -25.4
        43        45  9.02          0.77         -25.4
        39        44  9.00          0.76         -25.4
        38        44  9.00          0.76         -25.4
        42        44  8.98          0.76         -25.4
        21        48  8.97          0.80         -20.0
        43        46  8.97          0.76         -25.4
        21        47  8.95          0.80         -20.0


In [6]:
results_df.sort_values(by='CAGR', ascending=False).head(10)

,MA_Rapide,MA_Lente,CAGR,Sharpe_Ratio,Max_Drawdown
0,43,44,9.113462,0.772759,-25.400591
1,43,47,9.044178,0.766530,-25.400591
2,43,48,9.022167,0.764204,-25.400591
3,43,45,9.021057,0.765653,-25.400591
4,39,44,8.995235,0.764824,-25.400591
5,38,44,8.995235,0.764824,-25.400591
6,42,44,8.977785,0.763438,-25.400591
7,21,48,8.967549,0.797249,-20.001050
8,43,46,8.966690,0.761062,-25.400591
9,21,47,8.946823,0.796309,-20.001050


In [7]:
results_df.sort_values(by='Sharpe_Ratio', ascending=False).head(10)

,MA_Rapide,MA_Lente,CAGR,Sharpe_Ratio,Max_Drawdown
100,13,47,8.627783,0.824276,-17.265363
105,13,45,8.606613,0.823211,-17.265363
110,13,44,8.586230,0.822244,-17.265363
106,13,48,8.604933,0.821572,-17.265363
122,13,43,8.565943,0.821281,-17.265363
136,13,42,8.546402,0.820320,-17.265363
146,13,41,8.526302,0.819363,-17.265363
153,13,40,8.506941,0.818410,-17.265363
133,13,46,8.551539,0.817977,-17.265363
161,13,39,8.487027,0.817460,-17.265363
